## 오차역전파법

#### 계산 그래프
1. 계산 그래프를 구성
2. 그래프에서 계산을 왼쪽에서 오른쪽으로 진행 (순전파)

### 연쇄법칙
* 역전파는 국소적인 미분을 순방향과는 반대인 오른쪽에서 왼쪽으로 전달

#### 계산 그래프의 역전파
* 역전파의 계산 절차는 신호 E에 노드의 국소적 미분을 곱한 후 다음 노드로 전달하는 것
* 미분 값을 효율적으로 구할 수 있음

#### 연쇄법칙
* 합성 함수의 미분은 합성 함수를 구성하는 각 함수의 미분의 곱으로 나타낼 수 있음
$$
\frac{\partial z}{\partial x} = \frac{\partial z}{\partial t} \frac{\partial t}{\partial x}
$$

#### 덧셈 노드의 역전파
* z = x + y를 식으로 역전파를 살펴보면
$$
\frac{\partial z}{\partial x} = 1, \frac{\partial z}{\partial y} = 1
$$
* 즉 상류에서 전해진 미분에 1을 곱해 하류로 흘림

#### 곱셈 노드의 역전파
* z = xy로 보면
$$
\frac{\partial z}{\partial x} = y, \frac{\partial z}{\partial y} = x
$$
* 곱셈 노드 역전파는 상류의 값에 순전파 때의 입력 신호들을 서로 바꾼 값을 곱해서 하류로 보냄

### 단순한 계층 구현
* 곱셈 노드를 MulLayer, 덧셈 노드를 AddLayer로 구현

#### 곱셈 계층

In [1]:
class MulLayer:
    def __init__(self):
        self.x = None
        self.y = None

    def forward(self, x, y):
        self.x = x
        self.y = y
        out = x * y

        return out
    
    def backward(self, dout):
        dx = dout * self.y
        dy = dout * self.x

        return dx, dy

In [4]:
apple = 100
apple_num = 2
tax = 1.1

# 계층들
mul_apple_layer = MulLayer()
mul_tax_layer = MulLayer()

# 순전파
apple_price = mul_apple_layer.forward(apple, apple_num)
price = mul_tax_layer.forward(apple_price, tax)

print(price)

220.00000000000003


In [6]:
# 역전파
dprice = 1
dapple_price, dtax = mul_tax_layer.backward(dprice)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)

print(dapple, dapple_num, dtax)

2.2 110.00000000000001 200


#### 덧셈 계층

In [7]:
class AddLayer:
    def __init__(self):
        pass

    def forward(self, x, y):
        out = x + y
        return out
    
    def backward(self, dout):
        dx = dout * 1
        dy = dout * 1
        return dx, dy

In [8]:
# 덧셈 계층과 곱셈 계층을 활용해 사과 2개와 귤 3개 구입
apple = 100
apple_num = 2
orange = 150
orange_num = 3
tax = 1.1

# 계층들
mul_apple_layer = MulLayer()
mul_orange_layer = MulLayer()
add_apple_orange_layer = AddLayer()
mul_tax_layer = MulLayer()

# 순전파
apple_price = mul_apple_layer.forward(apple, apple_num)
orange_price = mul_orange_layer.forward(orange, orange_num)
all_price = add_apple_orange_layer.forward(apple_price, orange_price)
price = mul_tax_layer.forward(all_price, tax)

# 역전파
dprice = 1
dall_price, dtax = mul_tax_layer.backward(dprice)
dapple_price, dorange_price = add_apple_orange_layer.backward(dall_price)
dorange, dorange_num = mul_orange_layer.backward(dorange_price)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)

print(price)
print(dapple_num, dapple, dorange, dorange_num, dtax)

715.0000000000001
110.00000000000001 2.2 3.3000000000000003 165.0 650


### 활성화 함수 계층 구현

#### ReLU
$$
y =
\begin{cases}
x & (x > 0) \\
0 & (x \le 0)
\end{cases}
$$
$$
\frac{\partial y}{\partial x} =
\begin{cases}
1 & (x > 0) \\
0 & (x \le 0)
\end{cases}
$$

In [9]:
class Relu:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0

        return out

    def backward(self, dout):
        dout[self.mask] = 0
        dx = dout

        return dx

In [13]:
x = np.array([[1.0, -0.5], [-2.0, 3.0]])
print(x)

mask = (x <= 0)
print(mask)

[[ 1.  -0.5]
 [-2.   3. ]]
[[False  True]
 [ True False]]


#### Sigmoid 계층
$$
y = \frac{1}{1 + \exp(-x)}
$$

1. /노드, 즉 $y=\frac{1}{x}$ 를 미분하면
$$
\frac{\partial y}{\partial x} = -\frac{1}{x^2} = -y^2
$$
2. +노드에서는 그대로 전달
3. exp노드, $y=\exp(x)$ 연산을 수행
$$
\frac{\partial y}{\partial x} = \exp(x)
$$
4. *노드는 순전파 때의 값을 바꿔 곱함

* 정리
$$
\frac{\partial L}{\partial y}y^2 \exp(-x) = \frac{\partial L}{\partial y}y(1-y)
$$
* 시그모이드는 순전히 y 값만으로 계산이 가능

In [17]:
class Sigmoid:
    def __init__(self):
        self.out = None

    def forward(self, x):
        out = 1 / (1 + np.exp(-x))
        self.out = out
        return out

    def backward(self, dout):
        dx = dout * (1.0 - self.out) * self.out

        return dx

### Affine/Softmax 계층 구현

#### Affine 계층
* 신경망의 순전파 때 수행하는 행렬의 내적은 기하학에서 어파인 변환이라 함
* 어파인 변환을 수행하는 처리를 Affine 계층으로 구현

In [2]:
X = np.random.rand(2) # 입력
W = np.random.rand(2, 3) # 가중치
B = np.random.rand(3) # 편향

print(X.shape)
print(W.shape)
print(B.shape)

Y = np.dot(X, W) + B
print(Y)

(2,)
(2, 3)
(3,)
[1.19438235 1.10189421 1.55445232]


* 행렬을 사용한 역전파
$$
\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} \cdot W^T
$$
$$
\frac{\partial L}{\partial W} = X^T \cdot \frac{\partial L}{\partial Y}
$$

#### 배치용 Affine 계층
* 데이터 N개를 묶어 순전파하는 경우

In [9]:
X_dot_W = np.array([[0, 0, 0], [10, 10, 10]])
B = np.array([1, 2, 3])

print(X_dot_W)
print(X_dot_W + B)

[[ 0  0  0]
 [10 10 10]]
[[ 1  2  3]
 [11 12 13]]


In [13]:
dY = np.array([[1, 2, 3], [4, 5, 6]])
print(dY)

dB = np.sum(dY, axis=0)
print(dB)

[[1 2 3]
 [4 5 6]]
[5 7 9]


In [14]:
class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        out = np.dot(self.x, self.W) + self.b

        return out

    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        return dx

In [19]:
affine = Affine(W, B)

print(affine.forward(X))
print(affine.backward(1))
print(affine.dW, affine.db)

[1.45997272 2.36868827 3.76853952]
[[0.13066506 0.63873743]
 [0.01365656 0.66880695]
 [0.65967846 0.30722642]]
[0.91700703 0.53253798] 1


#### Softmax-with-Loss 계층
* 소프트맥스는 입력 값을 정규화하여 출력
* MNIST의 손글씨 숫자는 가짓수가 10개이므로 Softmax 계층의 입력은 10개
$$
\frac{\partial L}{\partial a_i} = y_i - t_i
$$

* 예시
    * t = (0, 1, 0)인 경우 softmax가 (0.3, 0.2, 0.5)를 출력한 경우
    * Softmax 계층의 역전파는 (0.3, -0.8, 0.5)가 됨
    * Softmax 계층의 앞 계층들은 큰 오차로부터 큰 깨달음을 얻게 됨

In [20]:
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None # 손실
        self.y = None    # softmax의 출력
        self.t = None    # 정답 레이블(원-핫 벡터)
        
    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)
        
        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        dx = (self.y - self.t) / batch_size
        
        return dx


### 오차역전파법 구현
* 전제
    * 신경망에는 적응 가능한 가중치와 편향이 있고, 이 가중치와 편향을 훈련 데이터에 적응하도록 조정하는 과정을 학습이라 함
    * 4단계로 수행
1. 미니배치
    * 훈련 데이터 중 일부를 무작위로 가져옴
    * 선별한 데이터를 미니배치라 하며, 미니배치의 손실 함수 값을 줄이는 것을 목표로 함
2. 기울기 산출
    * 미니배치의 손실 함수 값을 줄이기 위해 각 가중치 매개변수의 기울기를 구함
    * 기울기는 손실 함수의 값을 가장 작게 하는 방향을 제시
3. 매개변수 갱신
    * 가중치 매개변수를 기울기 바향으로 아주 조금 갱신
4. 반복
    * 1~3단계를 반복


In [ ]:
import sys, os
sys.path.append(os.pardir)
import numpy as np
from common.functions import *
from common.gradient import numerical_gradient
from collections import OrderedDict


class TwoLayerNet:

    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        # 가중치 초기화
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

        self.layers = OrderedDict()
        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1'])
        self.layers['Relu1'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2'])

        self.lastLayer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)

        return x

    def loss(self, x, t):
        y = self.predict(x)
        return self.lastLayer.forward(y, t)
        
    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)
        if t.ndim != 1 : t = np.argmax(t, axis=1)
        
        accuracy = np.sum(y == t) / float(x.shape[0])
        return accuracy
        

    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)
        
        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])
        
        return grads
        
    def gradient(self, x, t):
        # 순전파
        self.loss(x, t)

        # 역전파
        dout = 1
        dout = self.lastLayer.backward(dout)

        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        # 결과 저장
        grads = {}
        grads['W1'] = self.layers['Affine1'].dW
        grads['b1'] = self.layers['Affine1'].db
        grads['W2'] = self.layers['Affine2'].dW
        grads['b2'] = self.layers['Affine2'].db
        
        return grads

#### 기울기 검증

In [ ]:
import sys, os
sys.path.append(os.pardir)
import numpy as np
from dataset.mnist import load_mnist

# 데이터 읽기
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

x_batch = x_train[:3]
t_batch = t_train[:3]

grad_numerical = network.numerical_gradient(x_batch, t_batch)
grad_backprop = network.gradient(x_batch, t_batch)

# 각 가중치의 절대 오차의 평균을 구한다
for key in grad_numerical.keys():
    diff = np.average( np.abs(grad_backprop[key] - grad_numerical[key]) )
    print(key + ":" + str(diff))

W1:1.9122224061400168e-10
b1:9.95776553261202e-10
W2:6.899910500998696e-08
b2:1.3758571119765194e-07


* 수치 미분과 오차역전파법으로 구한 기울기의 차이가 매우 작음

#### 오차역전파법을 사용한 학습 구현하기

In [ ]:
import sys, os
sys.path.append(os.pardir)

import numpy as np
from dataset.mnist import load_mnist

# 데이터 읽기
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

iters_num = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]
    
    # 기울기 계산
    grad = network.gradient(x_batch, t_batch)
    
    # 갱신
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate * grad[key]
    
    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)
    
    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print(train_acc, test_acc)

0.09915 0.1009
0.80355 0.8065
0.8787 0.8826
0.89855 0.9016
0.9075333333333333 0.9113
0.9125166666666666 0.9159
0.9182 0.9206
0.92235 0.9252
0.9253166666666667 0.9273
0.92865 0.9316
0.9313333333333333 0.9324
0.93365 0.9352
0.9362166666666667 0.9362
0.9387333333333333 0.94
0.9409666666666666 0.9405
0.9426833333333333 0.9425
0.9451833333333334 0.9433
